# 01b · Datos de OpenStreetMap (Hugo)

No llama a Overpass: lee archivos con `src/fuente_osm.py`.

* `PATRULLA_OSM=local` (por defecto): archivos reales en `data/osm/`. La salida es `data/processed/areas_distritos.csv`, la que necesitan Flavia y Deivis.
* `PATRULLA_OSM=mock`: datos **inventados** en `data/osm_mock/`, solo para programar. La salida es `areas_distritos_MOCK.csv` y se queda en esa carpeta.

In [1]:
import sys
import pandas as pd

sys.path.insert(0, "../src")
import fuente_osm as fuente

if fuente.es_mock():
    import mock_osm
    print("limites, lugares, nodos, aristas:", mock_osm.generar())
    print("ATENCIÓN: DATOS SIMULADOS. No usar en el informe.")
print("Fuente:", fuente.fuente(), "| Carpeta:", fuente.carpeta())

Fuente: local | Carpeta: D:\QUINTO CICLO\IA\PROYECTO IA\data\osm


## 1. Límites de los 43 distritos y área

In [2]:
limites = fuente.cargar_limites()
areas = limites[["UBIGEO", "DIST_NORM"]].copy()
areas["AREA_KM2"] = limites.to_crs(fuente.CRS_METROS).area / 1e6

assert len(areas) == 43, "No hay 43 distritos"
assert areas["UBIGEO"].is_unique
assert (areas["AREA_KM2"] > 0).all()
areas.head()

,UBIGEO,DIST_NORM,AREA_KM2
0,150101,LIMA,21.532155
1,150102,ANCON,324.898802
2,150103,ATE,84.599068
3,150104,BARRANCO,2.809153
4,150105,BRENA,3.215212


## 2. Lugares por categoría y distrito

In [3]:
lugares = fuente.cargar_lugares(limites)
assert lugares["UBIGEO"].nunique() == 43, "Hay distritos sin ningún lugar"
lugares.pivot_table(index="UBIGEO", columns="CATEGORIA", aggfunc="size", fill_value=0).head()

CATEGORIA,banco_cajero,bar,comisaria,mercado,paradero
UBIGEO,,,,,
150101,103,29,26,52,110
150102,1,1,3,8,33
150103,40,5,7,33,39
150104,14,36,3,3,16
150105,20,2,3,12,6


## 3. Red de calles

In [4]:
G = fuente.cargar_calles()
aristas = fuente.ox.graph_to_gdfs(G, nodes=False)[["length", "geometry"] + (["lit"] if any("lit" in d for _, _, d in G.edges(data=True)) else [])]
print("Nodos:", G.number_of_nodes(), "| Aristas:", G.number_of_edges())

Nodos: 137896 | Aristas: 360845


## 4. Comprobaciones en los distritos piloto
Si muy pocas calles tienen `lit`, se quita ese factor. Con datos simulados estos números no significan nada; con datos reales, las comisarías se comparan a mano con la lista de la PNP.

In [5]:
PILOTOS = ["150101", "150132"]
import geopandas as gpd

aristas_m = aristas.to_crs(fuente.CRS_METROS)
aristas_m["geometry"] = aristas_m.geometry.representative_point()
por_distrito = gpd.sjoin(aristas_m, limites.to_crs(fuente.CRS_METROS)[["UBIGEO", "geometry"]], predicate="within")

for u in PILOTOS:
    tramo = por_distrito[por_distrito["UBIGEO"] == u]
    con_lit = tramo["lit"].notna().mean() if "lit" in tramo else 0
    comisarias = ((lugares["UBIGEO"] == u) & (lugares["CATEGORIA"] == "comisaria")).sum()
    print(f"{u}: {len(tramo)} tramos | con etiqueta lit: {con_lit:.1%} | comisarías: {comisarias}")

150101: 5530 tramos | con etiqueta lit: 3.7% | comisarías: 26
150132: 37965 tramos | con etiqueta lit: 2.2% | comisarías: 24


## 5. Guardar áreas

In [6]:
salida = fuente.ruta_areas()
areas.to_csv(salida, index=False)
print("Guardado:", salida.resolve(), areas.shape)

Guardado: D:\QUINTO CICLO\IA\PROYECTO IA\data\processed\areas_distritos.csv (43, 3)
